In [ ]:
!pip install -q openai

from google.colab import userdata
from openai import OpenAI

In [ ]:
client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a friendly restaurant assistant."},
        {"role": "user", "content": "Recommend a good vegetarian meal."}
    ],
)

print(response.choices[0].message.content)

In [ ]:
SYSTEM = """ROLE: You are the assistant for TastyBites Restaurant.

FACTS: We serve vegetarian and non-vegetarian meals, snacks, and beverages.
       Orders depend on menu and stock availability.
       Dine-in, takeaway, and delivery options may be available.
       Prices are listed in the restaurant menu dictionary.

RULES: Recommend food based on customer preferences.
       Ask about dietary needs or allergies when relevant.
       Never invent menu items, prices, or availability.
       Do not claim an order is placed unless the ordering system confirms it.
       Ask questions if order details are unclear.
       If you don't know, say "I'll check with the restaurant."
       Provide clear information about menu items and order totals.

FORMAT: Reply politely in under 60 words.
        Use friendly English or Telugu matching the customer."""

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "Can you recommend a meal for a student?"}
    ],
)

print(response.choices[0].message.content)
print(response.usage)

In [ ]:
MENU = {
    "veg biryani": 180,
    "paneer butter masala": 220,
    "chicken biryani": 240,
    "masala dosa": 80,
    "veg fried rice": 150,
    "chicken noodles": 190,
    "gulab jamun": 60,
    "fresh lime soda": 50
}

MENU["veg biryani"]

MENU.get("pizza", "Item not available")

In [ ]:
order = {
    "name": "Ravi",
    "item": "veg biryani",
    "qty": 2
}

order["qty"] * MENU[order["item"]]

In [ ]:
import json

text = json.dumps(order)

back = json.loads(text)

text

In [ ]:
back

In [ ]:
def get_price(item: str, qty: int = 1) -> int:
    item = item.lower().strip()
    if item not in MENU:
        raise ValueError(f"{item} is not available on the menu.")
    if qty <= 0:
        raise ValueError("Quantity must be greater than zero.")
    return MENU[item] * qty


get_price("veg biryani", 2)
get_price("Masala Dosa")

In [ ]:
import functools

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"[log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper


@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for selected menu items."""
    item = item.lower().strip()
    if item not in MENU:
        raise ValueError(f"{item} is not available on the menu.")
    if qty <= 0:
        raise ValueError("Quantity must be greater than zero.")
    return MENU[item] * qty


get_price("veg biryani", 2)
get_price("Masala Dosa")

In [ ]:
from openai import RateLimitError, APIConnectionError

history = [{"role": "system", "content": SYSTEM}]

MAX_TURNS = 8

def chat(user_text: str) -> str:

    history.append({"role": "user", "content": user_text})

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content or "Sorry, I couldn't prepare a response."

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."

    except Exception as e:
        reply = f"Something went wrong: {e}"

    history.append({"role": "assistant", "content": reply})

    return reply

In [ ]:
while True:

    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for contacting TastyBites Restaurant!")
        break

    print("Bot:", chat(text))

In [ ]:
history